# E-commerce Delivery Intelligence

## Data Preparation

This notebook prepares the Olist e-commerce data for delivery-performance analysis and predictive modelling.

The objective is to create a clean analytical dataset where each row represents one order.

In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/olist_orders_dataset.csv")
customers = pd.read_csv("../data/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
category_translation = pd.read_csv("../data/product_category_name_translation.csv")

In [3]:
orders.shape

(99441, 8)

In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


In [5]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


In [7]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

In [8]:
delivered_orders.shape

(96478, 8)

In [9]:
delivered_orders[
    "order_delivered_customer_date"
].isna().sum()

np.int64(8)

In [10]:
delivered_orders = delivered_orders.dropna(
    subset=[
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

In [11]:
delivered_orders.shape

(96470, 8)

In [12]:
delivered_orders["is_late"] = (
    delivered_orders["order_delivered_customer_date"]
    > delivered_orders["order_estimated_delivery_date"]
).astype(int)

In [13]:
delivered_orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "is_late"
    ]
].head(10)

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,0
5,2017-07-09 21:57:05,2017-07-26 10:57:55,2017-08-01,0
7,2017-05-16 13:10:30,2017-05-26 12:55:51,2017-06-07,0
8,2017-01-23 18:29:09,2017-02-02 14:08:10,2017-03-06,0
9,2017-07-29 11:55:02,2017-08-16 17:14:30,2017-08-23,0
10,2017-05-16 19:41:10,2017-05-29 11:18:31,2017-06-07,0


In [14]:
delivered_orders["is_late"].value_counts()

is_late
0    88644
1     7826
Name: count, dtype: int64

In [15]:
delivered_orders["is_late"].mean()

np.float64(0.08112366538820359)

In [16]:
late_rate = delivered_orders["is_late"].mean() * 100

print(f"Late delivery rate: {late_rate:.2f}%")

Late delivery rate: 8.11%


In [17]:
delivered_orders["actual_delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [18]:
delivered_orders[
    ["order_purchase_timestamp",
     "order_delivered_customer_date",
     "actual_delivery_days"]
].head()

,order_purchase_timestamp,order_delivered_customer_date,actual_delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.436574
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.782037
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.394213
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.208750
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.873877


In [19]:
delivered_orders["promised_delivery_days"] = (
    delivered_orders["order_estimated_delivery_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [20]:
delivered_orders[
    ["order_purchase_timestamp",
     "order_estimated_delivery_date",
     "promised_delivery_days"]
].head()

,order_purchase_timestamp,order_estimated_delivery_date,promised_delivery_days
0,2017-10-02 10:56:33,2017-10-18,15.544063
1,2018-07-24 20:41:37,2018-08-13,19.137766
2,2018-08-08 08:38:49,2018-09-04,26.639711
3,2017-11-18 19:28:06,2017-12-15,26.188819
4,2018-02-13 21:18:39,2018-02-26,12.112049


In [21]:
delivered_orders["purchase_month"] = (
    delivered_orders["order_purchase_timestamp"].dt.month
)

In [22]:
delivered_orders[
    ["order_purchase_timestamp", "purchase_month"]
].head()

,order_purchase_timestamp,purchase_month
0,2017-10-02 10:56:33,10
1,2018-07-24 20:41:37,7
2,2018-08-08 08:38:49,8
3,2017-11-18 19:28:06,11
4,2018-02-13 21:18:39,2


In [23]:
delivered_orders["purchase_weekday"] = (
    delivered_orders["order_purchase_timestamp"].dt.day_name()
)

In [24]:
delivered_orders[
    ["order_purchase_timestamp", "purchase_weekday"]
].head()

,order_purchase_timestamp,purchase_weekday
0,2017-10-02 10:56:33,Monday
1,2018-07-24 20:41:37,Tuesday
2,2018-08-08 08:38:49,Wednesday
3,2017-11-18 19:28:06,Saturday
4,2018-02-13 21:18:39,Tuesday


In [25]:
delivered_orders["purchase_year_month"] = (
    delivered_orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

In [26]:
delivered_orders[
    ["order_purchase_timestamp", "purchase_year_month"]
].head()

,order_purchase_timestamp,purchase_year_month
0,2017-10-02 10:56:33,2017-10
1,2018-07-24 20:41:37,2018-07
2,2018-08-08 08:38:49,2018-08
3,2017-11-18 19:28:06,2017-11
4,2018-02-13 21:18:39,2018-02


In [27]:
delivered_orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'is_late',
 'actual_delivery_days',
 'promised_delivery_days',
 'purchase_month',
 'purchase_weekday',
 'purchase_year_month']

In [28]:
delivered_orders[
    [
        "actual_delivery_days",
        "promised_delivery_days"
    ]
].describe()

,actual_delivery_days,promised_delivery_days
count,96470.000000,96470.000000
mean,12.558217,23.736343
std,9.546156,8.761052
min,0.533414,2.008009
25%,6.766204,18.329905
50%,10.217477,23.230880
75%,15.720182,28.407795
max,209.628611,155.135463


### Delivery Features

For delivery-performance analysis, the dataset was restricted to completed deliveries with valid actual and estimated delivery dates.

Several features were engineered:

- `is_late`: identifies orders delivered after their estimated delivery date.
- `actual_delivery_days`: measures the time between purchase and actual delivery.
- `promised_delivery_days`: measures the delivery window originally provided to the customer.
- `purchase_month`: captures potential seasonal effects.
- `purchase_weekday`: captures potential differences by order day.
- `purchase_year_month`: supports time-series analysis.

`actual_delivery_days` will be used for descriptive analysis only and not as a predictive-model input, because the actual delivery duration would not be known when an order is placed.

In [29]:
item_summary = (
    order_items
    .groupby("order_id")
    .agg(
        order_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        number_of_items=("order_item_id", "count")
    )
    .reset_index()
)

In [30]:
item_summary.head()

,order_id,order_value,freight_value,number_of_items
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1


In [31]:
item_summary["order_id"].is_unique

True

In [32]:
master = delivered_orders.merge(
    item_summary,
    on="order_id",
    how="left"
)

In [33]:
master.shape

(96470, 17)

In [34]:
delivered_orders.shape

(96470, 14)

In [35]:
master["order_id"].is_unique

True

In [36]:
customer_info = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
]

In [37]:
master = master.merge(
    customer_info,
    on="customer_id",
    how="left"
)

In [38]:
master.shape

(96470, 20)

In [39]:
master["order_id"].is_unique

True

## Enriching the Order-Level Dataset

The remaining datasets contain product, seller, payment and review information. Because some orders contain multiple products, sellers, payments or reviews, these tables will be aggregated before being joined to the order-level dataset.

In [40]:
item_details = order_items.merge(
    products[
        ["product_id", "product_category_name"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [41]:
item_details.shape

(112650, 8)

In [42]:
order_items.shape

(112650, 7)

In [43]:
item_details = item_details.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

In [44]:
item_details[
    [
        "product_category_name",
        "product_category_name_english"
    ]
].head(10)

,product_category_name,product_category_name_english
0,cool_stuff,cool_stuff
1,pet_shop,pet_shop
2,moveis_decoracao,furniture_decor
3,perfumaria,perfumery
4,ferramentas_jardim,garden_tools
5,utilidades_domesticas,housewares
6,telefonia,telephony
7,ferramentas_jardim,garden_tools
8,beleza_saude,health_beauty
9,livros_tecnicos,books_technical


In [45]:
item_details["product_category_name_english"] = (
    item_details["product_category_name_english"]
    .fillna("unknown")
)

In [46]:
item_details = item_details.merge(
    sellers[
        ["seller_id", "seller_city", "seller_state"]
    ],
    on="seller_id",
    how="left",
    validate="many_to_one"
)

In [47]:
item_details[
    [
        "order_id",
        "product_category_name_english",
        "seller_state",
        "price"
    ]
].head(10)

,order_id,product_category_name_english,seller_state,price
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,SP,58.90
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,SP,239.90
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,MG,199.00
3,00024acbcdf0a6daa1e931b038114c75,perfumery,SP,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,PR,199.90
5,00048cc3ae777c65dbb7d2a0634bc1ea,housewares,SP,21.90
6,00054e8431b9d7675808bcb819fb4a32,telephony,SP,19.90
7,000576fe39319847cbb9d288c5617fa6,garden_tools,SP,810.00
8,0005a1a1728c9d785b8e2b08b904576c,health_beauty,SP,145.95
9,0005f50442cb953dcd1d21e1fb923495,books_technical,SP,53.99


In [48]:
category_value = (
    item_details
    .groupby(
        ["order_id", "product_category_name_english"],
        as_index=False
    )["price"]
    .sum()
)

In [49]:
category_value.head()

,order_id,product_category_name_english,price
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,58.90
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,239.90
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,199.00
3,00024acbcdf0a6daa1e931b038114c75,perfumery,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,199.90


In [50]:
primary_category = (
    category_value
    .sort_values(
        ["order_id", "price"],
        ascending=[True, False]
    )
    .drop_duplicates("order_id")
    [["order_id", "product_category_name_english"]]
    .rename(
        columns={
            "product_category_name_english":
            "primary_product_category"
        }
    )
)

In [51]:
primary_category.head()

,order_id,primary_product_category
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools


In [52]:
primary_category["order_id"].is_unique

True

In [53]:
seller_value = (
    item_details
    .groupby(
        ["order_id", "seller_state"],
        as_index=False
    )["price"]
    .sum()
)

In [54]:
primary_seller = (
    seller_value
    .sort_values(
        ["order_id", "price"],
        ascending=[True, False]
    )
    .drop_duplicates("order_id")
    [["order_id", "seller_state"]]
    .rename(
        columns={
            "seller_state": "primary_seller_state"
        }
    )
)

In [55]:
primary_seller = (
    seller_value
    .sort_values(
        ["order_id", "price"],
        ascending=[True, False]
    )
    .drop_duplicates("order_id")
    [["order_id", "seller_state"]]
    .rename(
        columns={
            "seller_state": "primary_seller_state"
        }
    )
)

In [56]:
primary_seller["order_id"].is_unique

True

In [57]:
primary_seller.head()

,order_id,primary_seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,SP
1,00018f77f2f0320c557190d7a144bdd3,SP
2,000229ec398224ef6ca0657da4fc703e,MG
3,00024acbcdf0a6daa1e931b038114c75,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,PR


In [59]:
item_complexity = (
    item_details
    .groupby("order_id")
    .agg(
        number_of_categories=(
            "product_category_name_english",
            "nunique"
        ),
        number_of_sellers=(
            "seller_id",
            "nunique"
        )
    )
    .reset_index()
)

In [60]:
item_complexity.head()

,order_id,number_of_categories,number_of_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,1
2,000229ec398224ef6ca0657da4fc703e,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1


In [61]:
len(master)

96470

In [62]:
master = master.merge(
    primary_category,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [63]:
master = master.merge(
    primary_seller,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [64]:
master = master.merge(
    item_complexity,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [65]:
master.shape

(96470, 24)

In [66]:
master["order_id"].is_unique

True

In [67]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [68]:
payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=(
            "payment_value",
            "sum"
        ),
        max_payment_installments=(
            "payment_installments",
            "max"
        ),
        number_of_payment_records=(
            "payment_sequential",
            "count"
        )
    )
    .reset_index()
)

In [69]:
payment_summary.head()

,order_id,total_payment_value,max_payment_installments,number_of_payment_records
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1


In [70]:
payment_summary["order_id"].is_unique

True

In [71]:
primary_payment = (
    payments
    .sort_values(
        ["order_id", "payment_value"],
        ascending=[True, False]
    )
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(
        columns={
            "payment_type": "primary_payment_type"
        }
    )
)

In [72]:
primary_payment.head()

,order_id,primary_payment_type
85283,00010242fe8c5a6d1ba2dd792cb16214,credit_card
2499,00018f77f2f0320c557190d7a144bdd3,credit_card
12393,000229ec398224ef6ca0657da4fc703e,credit_card
32971,00024acbcdf0a6daa1e931b038114c75,credit_card
98711,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card


In [73]:
primary_payment.head()

,order_id,primary_payment_type
85283,00010242fe8c5a6d1ba2dd792cb16214,credit_card
2499,00018f77f2f0320c557190d7a144bdd3,credit_card
12393,000229ec398224ef6ca0657da4fc703e,credit_card
32971,00024acbcdf0a6daa1e931b038114c75,credit_card
98711,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card


In [74]:
master = master.merge(
    primary_payment,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [75]:
master["order_id"].is_unique

True

In [76]:
master.shape

(96470, 25)

In [77]:
review_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=(
            "review_score",
            "mean"
        ),
        number_of_reviews=(
            "review_id",
            "count"
        )
    )
    .reset_index()
)

In [78]:
review_summary.head()

,order_id,review_score,number_of_reviews
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [79]:
review_summary["order_id"].is_unique

True

In [80]:
master = master.merge(
    review_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [81]:
master.shape

(96470, 27)

In [82]:
master["order_id"].is_unique

True

In [84]:
master["order_id"].duplicated().sum()

np.int64(0)

In [85]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,actual_delivery_days,...,customer_unique_id,customer_city,customer_state,primary_product_category,primary_seller_state,number_of_categories,number_of_sellers,primary_payment_type,review_score,number_of_reviews
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,8.436574,...,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,housewares,SP,1,1,voucher,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,13.782037,...,af07308b275d755c9edb36a90c618231,barreiras,BA,perfumery,SP,1,1,boleto,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9.394213,...,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,auto,SP,1,1,credit_card,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,13.208750,...,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,pet_shop,MG,1,1,credit_card,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,2.873877,...,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,stationery,SP,1,1,credit_card,5.0,1.0


In [86]:
master.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'is_late',
 'actual_delivery_days',
 'promised_delivery_days',
 'purchase_month',
 'purchase_weekday',
 'purchase_year_month',
 'order_value',
 'freight_value',
 'number_of_items',
 'customer_unique_id',
 'customer_city',
 'customer_state',
 'primary_product_category',
 'primary_seller_state',
 'number_of_categories',
 'number_of_sellers',
 'primary_payment_type',
 'review_score',
 'number_of_reviews']

In [87]:
missing_summary = (
    master
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

number_of_reviews                646
review_score                     646
order_approved_at                 14
primary_payment_type               1
order_delivered_carrier_date       1
freight_value                      0
number_of_sellers                  0
number_of_categories               0
primary_seller_state               0
primary_product_category           0
customer_state                     0
customer_city                      0
customer_unique_id                 0
number_of_items                    0
order_id                           0
order_value                        0
customer_id                        0
purchase_weekday                   0
purchase_month                     0
promised_delivery_days             0
actual_delivery_days               0
is_late                            0
order_estimated_delivery_date      0
order_delivered_customer_date      0
order_purchase_timestamp           0
order_status                       0
purchase_year_month                0
d

In [88]:
missing_percentage = (
    master.isna().mean() * 100
).sort_values(ascending=False)

missing_percentage.head(15)

number_of_reviews               0.669638
review_score                    0.669638
order_approved_at               0.014512
primary_payment_type            0.001037
order_delivered_carrier_date    0.001037
freight_value                   0.000000
number_of_sellers               0.000000
number_of_categories            0.000000
primary_seller_state            0.000000
primary_product_category        0.000000
customer_state                  0.000000
customer_city                   0.000000
customer_unique_id              0.000000
number_of_items                 0.000000
order_id                        0.000000
dtype: float64

In [90]:
[col for col in master.columns if "payment" in col]

['primary_payment_type']

In [91]:
payment_summary.columns.tolist()

['order_id',
 'total_payment_value',
 'max_payment_installments',
 'number_of_payment_records']

In [92]:
[col for col in master.columns if "total_payment_value" in col]

[]

In [93]:
master = master.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [94]:
[col for col in master.columns if "payment" in col]

['primary_payment_type',
 'total_payment_value',
 'max_payment_installments',
 'number_of_payment_records']

In [95]:
master["order_id"].is_unique

True

In [96]:
master["order_id"].duplicated().sum()

np.int64(0)

In [97]:
master[
    [
        "is_late",
        "actual_delivery_days",
        "promised_delivery_days",
        "order_value",
        "freight_value",
        "number_of_items",
        "number_of_categories",
        "number_of_sellers",
        "total_payment_value",
        "max_payment_installments",
        "review_score"
    ]
].describe()

,is_late,actual_delivery_days,promised_delivery_days,order_value,freight_value,number_of_items,number_of_categories,number_of_sellers,total_payment_value,max_payment_installments,review_score
count,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96469.000000,96469.000000,95824.000000
mean,0.081124,12.558217,23.736343,137.040001,22.785798,1.142210,1.008272,1.013901,159.855320,2.928039,4.156158
std,0.273026,9.546156,8.761052,209.052608,21.559959,0.538824,0.092611,0.123542,218.820934,2.712802,1.283615
min,0.000000,0.533414,2.008009,0.850000,0.000000,1.000000,1.000000,1.000000,9.590000,0.000000,1.000000
25%,0.000000,6.766204,18.329905,45.900000,13.850000,1.000000,1.000000,1.000000,61.880000,1.000000,4.000000
50%,0.000000,10.217477,23.230880,86.500000,17.170000,1.000000,1.000000,1.000000,105.280000,2.000000,5.000000
75%,0.000000,15.720182,28.407795,149.900000,24.020000,1.000000,1.000000,1.000000,176.330000,4.000000,5.000000
max,1.000000,209.628611,155.135463,13440.000000,1794.960000,21.000000,3.000000,5.000000,13664.080000,24.000000,5.000000


### Order-Level Dataset

The relational Olist tables were aggregated and combined to create a master dataset with one row per delivered order.

Order-item data was aggregated to calculate order value, freight cost and item count. Product categories and seller locations were represented using the category and seller state contributing the greatest merchandise value to each order.

Payment records were aggregated to preserve one row per order, while the highest-value payment method was used as the primary payment type.

Customer reviews were aggregated separately and retained for customer-experience analysis. Review information will not be used as a predictor of delivery lateness because it would not be available at the time an order is placed.

After each merge, order-ID uniqueness and row counts were checked to ensure that one-to-many relationships did not unintentionally duplicate orders.